Firstly we will import all the dependencies 

In [16]:
#!pip install scikit-learn
#!pip install tensorflow

In [90]:
import json
import os
import shutil
import cv2
import numpy as np
import os
# from matplotlib import pyplot as plt
# import time
import mediapipe as mp
# import re

# import cv2
# import numpy as np
# import os

In [91]:
video_data_folder = "../archive"

numerical_data_path='../NumericalData_AUTSL'

In [92]:
mp_holistic = mp.solutions.holistic # Holistic model
# mp_drawing = mp.solutions.drawing_utils # Drawing utilities

In [93]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # COLOR CONVERSION BGR 2 RGB
    image.flags.writeable = False                  # Image is no longer writeable
    results = model.process(image)                 # Make prediction
    image.flags.writeable = True                   # Image is now writeable
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR) # COLOR COVERSION RGB 2 BGR
    return image, results

In [94]:
def extract_keypoints(results):
    pose = np.array([[res.x, res.y, res.z] for res in results.pose_landmarks.landmark]).flatten() if results.pose_landmarks else np.zeros(33*3) 
    face = np.array([[res.x, res.y, res.z] for res in results.face_landmarks.landmark]).flatten() if results.face_landmarks else np.zeros(468*3)
    lh = np.array([[res.x, res.y, res.z] for res in results.left_hand_landmarks.landmark]).flatten() if results.left_hand_landmarks else np.zeros(21*3)
    rh = np.array([[res.x, res.y, res.z] for res in results.right_hand_landmarks.landmark]).flatten() if results.right_hand_landmarks else np.zeros(21*3)
    return np.concatenate([pose, face, lh, rh])

In [95]:
def process_video(src_file_path, dest_folder, dest_file_name):

    dest_file_path=dest_folder+'/'+dest_file_name
    if os.path.exists(dest_file_path):
        return 
    
    print("making npy from video")
    cap = cv2.VideoCapture(src_file_path)
    # print(video_path)
    sequence_length = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sequence = 1
    # Set mediapipe model
    video_keypoints = []

    with mp_holistic.Holistic(min_detection_confidence=0.5, min_tracking_confidence=0.5) as holistic:
        # for sequence in range(no_sequences):
            for frame_num in range(sequence_length):
                ret, frame = cap.read()
                if not ret:
                    break

                # Make de8tections
                image, results = mediapipe_detection(frame, holistic)
               
                # Export keypoints
                keypoints = extract_keypoints(results)
                
                video_keypoints.append(keypoints)
                sequence += 1
                # Break gracefully
                #if cv2.waitKey(10) & 0xFF == ord('q'):
                #    break
    
    cap.release()
    
    np.save(dest_file_path, video_keypoints)

In [96]:
def makeNpy(source_path,file_name, classIndex, split,dest_path):
    
    dest_file_name =classIndex+'_'+file_name+'.npy'
    file_name=file_name+"_color.mp4"


    dest_path+='/'+split
    source_path+='/'+file_name
    # print(source_path)
    
    # print(dest_path)
    if os.path.exists(dest_path):
        pass
    else:
        os.makedirs(dest_path)
    process_video(source_path,dest_path,dest_file_name)

In [98]:
import sys
csvs={f'{video_data_folder}/test_labels.csv', f'{video_data_folder}/train_labels.csv',f'{video_data_folder}/val_labels.csv'}
for csv_path in csvs:
    if os.path.exists(csv_path):
        print(csv_path)
    else:
        sys.exit(0)
        


../archive/train_labels.csv
../archive/test_labels.csv
../archive/val_labels.csv


In [ ]:
import csv
dest_path=numerical_data_path
for csv_path in csvs:
    #parse the csav
    data_path = csv_path.split("_")[0]
    split=data_path.split("/")[-1]
    # print(split)
    # print(data_path)
    with open(csv_path, 'r') as file:
        data = csv.reader(file)
        for row in data:
            makeNpy(data_path,row[0],row[1],split,dest_path)

In [ ]:

import numpy as np
base_path='../NumericalData_AUTSL'
dir_list=os.listdir(base_path)
print(dir_list)


for dir in dir_list:
    dir_path=base_path+'/'+dir
    files=os.listdir(dir_path)
    # print(len(files))
    for file in files:
        file_path =dir_path+'/'+file
        # print(file_path)
        npy=np.load(file_path)
        if len(npy) ==0:          
            print(file_path)
            os.remove(file_path)

In [48]:
PER_FRAME_FEATURE =1629
MAX_FRAME =-1

In [49]:
max_frame =-9
min_frame =9999

dest_path=numerical_data_path
dirs=os.listdir(dest_path)

for dir in dirs:
    
    #parse the csav
    data_path = dest_path+'/'+dir
    trials=os.listdir(data_path)

    for trial in trials:
        trail_path=data_path+'/'+trial
        npy =np.load(trail_path)
        npy=npy.reshape(-1,1629)
        no_frame=npy.shape[0]
        if no_frame <min_frame:
            min_frame =no_frame
        if no_frame >max_frame:
            max_frame=no_frame


print('Max frame: ', max_frame)
print('min frame: ', min_frame)

MAX_FRAME=max_frame
    
            
    


Max frame:  156
min frame:  7


In [26]:
# npy = np.load('../NumericalData/train/26_signer0_sample4.npy')
# npy=npy.reshape(-1,1629)
# print(npy.shape)
# print(npy)

In [101]:
def makeRelative(npy_frame,  midshoulder_init_X,midshoulder_init_y,midshoulder_init_d):
    
    #if a point is missing do no calibration or relative translation on it

    #hand points will be translated relative to wrist point
    #wrist point will be ralative to elbow point
    #elbow point will be relative to shoulder point
    

    #shoulder points will be relatvie to their middle positions, this is already done during calibration with respect to first frame of the first sign of the video

    #face points will be relative to nose point
    #nose point will be relative to the middle of two shoulder points
    
    #@will not touch these foot parts as they are zeros in our case
    #heel and foot index point will be relative to ankle point
    #ankle point will be ralative to knee point
    #knee point will be relative to heap points 
    # heap points will be relative to the middle of two shoulder points, already done during calibratoin
    
    npy_mat=npy_frame.reshape(-1,3)

    point=542
    for i in range (0,543):
        x=npy_mat[point][0]
        y=npy_mat[point][1]
        d=npy_mat[point][2]
        
        if(x<=0.0 and y <=0.0 and d<=0.0) :
            point=point-1
            continue

        ref_x= 0
        ref_y= 0
        ref_d= 0
        if point in range (522,543): #right hand points
            ref_x= npy_mat[16][0] #ref right wrist
            ref_y= npy_mat[16][1]
            ref_d= npy_mat[16][2]
        elif point in range (501,522): #left hand
            ref_x= npy_mat[15][0] #ref left wrist
            ref_y= npy_mat[15][1]
            ref_d= npy_mat[15][2]
        elif point in range (33,501): #face points
            ref_x= npy_mat[0][0] # ref nose
            ref_y= npy_mat[0][1]
            ref_d= npy_mat[0][2]
        elif point in range (0,33):
            #point value comes from 32 to 0 as loop designed
            if point ==0: # nose   ref is middle of two shoulder
                ref_x= (npy_mat[11][0]+npy_mat[12][0])/2
                ref_y= (npy_mat[11][1]+npy_mat[12][1])/2
                ref_d= (npy_mat[11][2]+npy_mat[12][2])/2
            #as loop comes from 32 to 0, face points will come fist before nose value is translated            
            elif point in range (1,11): # face points ref is nose point
                ref_x= npy_mat[0][0] 
                ref_y= npy_mat[0][1]
                ref_d= npy_mat[0][2]
            elif point in range (11,13):  # two shoulder points do nothing as ref is initialized to zero first
                ref_x= 0
                ref_y= 0
                ref_d= 0
            #elbow and wrist comes before shoulder points as we loop 542 to 0
            elif point ==13: #lefthand elbow 
                #ref is left shoulder point 
                ref_x= npy_mat[11][0]
                ref_y= npy_mat[11][1]
                ref_d= npy_mat[11][2]

            elif point ==15: #lefthand wrist
                #ref is left elbow 
                ref_x= npy_mat[13][0]
                ref_y= npy_mat[13][1]
                ref_d= npy_mat[13][2]
            elif point ==14: #righht elbow 
                #ref is righht shoulder point 
                ref_x= npy_mat[12][0]
                ref_y= npy_mat[12][1]
                ref_d= npy_mat[12][2]

            elif point ==16: #righht wrist
                #ref is righht elbow 
                ref_x= npy_mat[14][0]
                ref_y= npy_mat[14][1]
                ref_d= npy_mat[14][2]
            # finger tips come before wrist as we loop from high to low
            elif point in range (17,23) and point %2 ==1: #lefthand  finger tips
                #ref is left wrist
                ref_x= npy_mat[15][0]
                ref_y= npy_mat[15][1]
                ref_d= npy_mat[15][2]
            elif point in range (17,23) and point %2 ==0: #righthand  finger tips
                #ref is right wrist
                ref_x= npy_mat[16][0]
                ref_y= npy_mat[16][1]
                ref_d= npy_mat[16][2]
            elif point in range (23,25):  # two hips do nothing
                ref_x= 0
                ref_y= 0
                ref_d= 0
            elif point ==25: #left kneeee 
                #ref is left hip 
                ref_x= npy_mat[23][0]
                ref_y= npy_mat[23][1]
                ref_d= npy_mat[23][2]

            elif point ==27: #left ankle
                #ref is left kneee
                ref_x= npy_mat[25][0]
                ref_y= npy_mat[25][1]
                ref_d= npy_mat[25][2]
            elif point ==29: #left heel
                #ref is left anklee
                ref_x= npy_mat[27][0]
                ref_y= npy_mat[27][1]
                ref_d= npy_mat[27][2]

            elif point ==31: #left index
                #ref is left anklee
                ref_x= npy_mat[27][0]
                ref_y= npy_mat[27][1]
                ref_d= npy_mat[27][2]


            elif point ==26: #rightt kneeee 
                #ref is righhtt hip 
                ref_x= npy_mat[24][0]
                ref_y= npy_mat[24][1]
                ref_d= npy_mat[24][2]

            elif point ==28: #righhtt ankle
                #ref is rightt kneee
                ref_x= npy_mat[26][0]
                ref_y= npy_mat[26][1]
                ref_d= npy_mat[26][2]

            elif point ==30: #righhtt heel
                #ref is rightt hip
                ref_x= npy_mat[28][0]
                ref_y= npy_mat[28][1]
                ref_d= npy_mat[28][2]

            elif point ==32: #righhtt index
                #ref is rightt hip
                ref_x= npy_mat[28][0]
                ref_y= npy_mat[28][1]
                ref_d= npy_mat[28][2]
            
        npy_mat[point][0] =x-ref_x
        npy_mat[point][1] =y-ref_y
        npy_mat[point][2] = d-ref_d
        point=point-1
    #for end
    
    for p in {11,13,23,25}:
        npy_mat[p][0] =npy_mat[p][0]-midshoulder_init_X
        npy_mat[p][1] =npy_mat[p][1]-midshoulder_init_y
        npy_mat[p][2] = npy_mat[p][2]-midshoulder_init_d




    return npy_mat.flatten()

In [105]:
def makeRelativeNpy(npy):
    npy=npy.reshape(-1,PER_FRAME_FEATURE)
    npy_points =npy[0].reshape(-1,3)
    mid_shoulder_x=(npy_points[11][0]+npy_points[13][0])/2
    mid_shoulder_y=(npy_points[11][1]+npy_points[13][1])/2
    mid_shoulder_d=(npy_points[11][2]+npy_points[13][2])/2
    for i in range(0,npy.shape[0]):
        temp=makeRelative(npy[i],mid_shoulder_x,mid_shoulder_y,mid_shoulder_d)
        npy[i]=temp
        
    return npy

In [106]:

base_path='../NumericalData_AUTSL'
dest_path_base='../AUTSL_RELATIVE'
dir_list=os.listdir(base_path)
print(dir_list)


for dir in dir_list:
    dir_path=base_path+'/'+dir

    files=os.listdir(dir_path)
    # print(len(files))
    for file in files:
        file_path =dir_path+'/'+file
        # print(file_path)
        npy=np.load(file_path)
        if len(npy) ==0:
            print("zero file: ",file_path)
        dest_path=dest_path_base+'/'+dir
        if os.path.exists(dest_path):
            pass
        else:
            os.makedirs(dest_path)
            
        dest_path=dest_path+'/'+file
        npy=makeRelativeNpy(npy)
        np.save(dest_path,npy)
        



['test', 'train', 'val']


[-0.01784337 -0.12299451 -0.39817975 ... -0.00077203  0.07737702
  0.42745841]
[ 0.55756879  0.34720755 -0.72362834 ...  0.44239348  0.84841776
  0.00362576]


In [114]:
#ZERO PADDING THE FRAMES
import numpy as np
base_path='../AUTSL_RELATIVE'
dest_base_path='../AUTSL_RELATIVE_ZEROPAD'
for dir in dir_list:
    dir_path=base_path+'/'+dir

    files=os.listdir(dir_path)
    # print(len(files))
    for file in files:
        file_path =dir_path+'/'+file
        
        # print(file_path)
        npy=np.load(file_path)
        npy_mat= npy.reshape(-1,PER_FRAME_FEATURE)
        no_frame = npy_mat.shape[0]
        diff = MAX_FRAME-no_frame

        list_npy_mat = []
        
        for i in range(0, no_frame):
            list_npy_mat.append(npy_mat[i])

        for i in range(0, diff):
            list_npy_mat.append(np.zeros(PER_FRAME_FEATURE))
        
        npy_new = np.array(list_npy_mat)

        dest_file_path=dest_base_path+'/'+dir

        if os.path.exists(dest_file_path):
            pass
        else:
            os.makedirs(dest_file_path)
        dest_file_path+='/'+file

        # print(npy_mat.shape)
        # print(npy_new.shape)
        np.save(dest_file_path,npy_new.flatten())




In [127]:
# npy=np.load('../NumericalData_RELATIVE/test/0_signer6_sample276.npy')
# npy=npy.reshape(-1,1629)
# print(npy[npy.shape[0]-1])

# npy=np.load('../NumericalData_RELATIVE_ZEROPAD/test/0_signer6_sample276.npy')
# npy=npy.reshape(-1,1629)
# print(npy[67])
# # npy=np.load('../NumericalData/test/0_signer6_sample276.npy')
# # npy=npy.reshape(-1,1629)
# # print(npy[0])


[-0.01457088 -0.1235176  -0.31041595 ...  0.00515339  0.07771027
  0.31952479]
[0. 0. 0. ... 0. 0. 0.]


In [123]:
#normalize the features
import os
import numpy as np


def normalizeData (norm_path, crossValidationDataPaths,testPaths):
    print("hello")
    # PER_FRAME_FEATURE =1629
    min=np.zeros (PER_FRAME_FEATURE)
    max=np.zeros (PER_FRAME_FEATURE)
    

    for i in range (0,PER_FRAME_FEATURE):
        min[i]= 9
        max[i]= -9

    ml_instances_paths=[]
    for path in crossValidationDataPaths:   
        trials= os.listdir(path)
        
        
        
        for trial in trials:
            trialPath =f'{path}/{trial}'
            ml_instances_paths.append(trialPath)           
            npy =np.load(trialPath)            
            npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
            feature =0
            for feature in range(0,PER_FRAME_FEATURE):
                temp_min = np.min(npy_matrix[:,feature])
                temp_max = np.max(npy_matrix[:,feature])
                
                if temp_max>max[feature]:
                    max[feature] =temp_max
                if temp_min < min[feature]:
                    min[feature] =temp_min
    
    test_paths=[]
    for path in testPaths:    
        
        trials= os.listdir(path)
        
        
            
        for trial in trials:
            trialPath =f'{path}/{trial}'            
            test_paths.append(trialPath)
            
            npy =np.load(trialPath)
            npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)          
            
            feature=0
            for feature in range(0,PER_FRAME_FEATURE):
                temp_min = np.min(npy_matrix[:,feature])
                temp_max = np.max(npy_matrix[:,feature])
                
                if temp_max>max[feature]:
                    max[feature] =temp_max
                if temp_min < min[feature]:
                    min[feature] =temp_min

    print('Normalization starting')

    for ml_instances_path in ml_instances_paths:         
        npy =np.load(ml_instances_path)
        npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
        feature=0
        for feature in range(0,PER_FRAME_FEATURE):          
            my_range = max[feature]-min[feature]        
            npy_matrix[:,feature] = (npy_matrix[:,feature]*100000 -min[feature]*100000)/(my_range*100000+0.00001)     
        
       
        label = ml_instances_path.split('/')
        dir_path =''
        i=0
        for i in range(2,len(label)-1): #omit ../
            dir_path=dir_path+label[i]+'/'

        dir_path=f'{norm_path}/{dir_path}'
        if os.path.exists(dir_path):
            pass           
                           
        else: 
            os.makedirs(dir_path)    

        toDest_path =f'{dir_path}/{label[len(label)-1]}'
        np.save(toDest_path,npy_matrix.flatten())

    for ml_instances_path in test_paths:  
        npy =np.load(ml_instances_path)
        npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
        feature =0
        for feature in range(0,PER_FRAME_FEATURE):
            my_range =max[feature]-min[feature]
            npy_matrix[:,feature] = (npy_matrix[:,feature]*100000 -min[feature]*100000)/(my_range*100000+0.00001)  

        #save in file
        # toDest_path=f'{norm_path}/{ml_instances_path}'

        label = ml_instances_path.split('/')
        dir_path =''
        i=0
        for i in range(2,len(label)-1):  #omit ../
            dir_path=dir_path+label[i]+'/'

        dir_path=f'{norm_path}/{dir_path}'
        if os.path.exists(dir_path):
            pass                                                   
        else: 
            os.makedirs(dir_path)
        
        toDest_path =f'{dir_path}/{label[len(label)-1]}'
        np.save(toDest_path,npy_matrix.flatten())

    



In [40]:
# crossValidationDataPaths={'../NumericalData/train','../NumericalData/val'}
# testPaths={'../NumericalData/test'}
# norm_path ='../NORMALIZED_AUTSL226'
# normalizeData (norm_path, crossValidationDataPaths,testPaths)

hello
Normalization starting


In [122]:
crossValidationDataPaths={'../AUTSL_RELATIVE_ZEROPAD/train','../AUTSL_RELATIVE_ZEROPAD/val'}
testPaths={'../AUTSL_RELATIVE_ZEROPAD/test'}
norm_path ='../NORMALIZED_RELATIVE_AUTSL226'
normalizeData (norm_path, crossValidationDataPaths,testPaths)

hello
Normalization starting


In [129]:
# npy=np.load('../NORMALIZED_RELATIVE_AUTSL226/test/0_signer6_sample276.npy')
# npy=npy.reshape(-1,1629)
# print(npy[66][11])
# print(npy[67][11])
# print(npy[69][11])

0.335247719118449
0.24593807410433202
0.24593807410433202


['test', 'train', 'val']


In [130]:
def quantizeFrame(npy1629):
    
    # decide quantization levels allowable for the point
    # get the quantization string for the point
    #concatanate all the quantizatioin strings to get the total string for the frame
    
    npy_mat =npy1629.reshape(-1,3)
    # print(npy_mat.shape)
    totalString=''
 

    point=542
    for i in range (0,543):
        x=npy_mat[point][0]
        y=npy_mat[point][1]
        d=npy_mat[point][2]
        
        
        if point in range (522,543): #right hand points
            x_level =10
            y_level=10
            d_level=5
            
            
        elif point in range (501,522): #left hand
            x_level =10
            y_level=10
            d_level=5
        elif point in range (33,501): #face points
            #special emphasis to be put for mount points..... will do later
            x_level =5
            y_level=5
            d_level=3

        elif point in range (0,33):
            #point value comes from 32 to 0 as loop designed
            if point ==0: # nose   
                x_level =5
                y_level=5
                d_level=3
            
            elif point in range (1,11): # face points in pose
                x_level =5
                y_level=5
                d_level=3
            elif point in range (11,13):  # two shoulder points 
                x_level =5
                y_level=5
                d_level=3

           
            elif point ==13: #lefthand elbow 
                x_level =10
                y_level=10
                d_level=5

            elif point ==15: #lefthand wrist
                x_level =10
                y_level=10
                d_level=5
            elif point ==14: #righht elbow 
                x_level =10
                y_level=10
                d_level=5

            elif point ==16: #righht wrist
                x_level =10
                y_level=10
                d_level=5
                
            elif point in range (17,23) and point %2 ==1: #lefthand  finger tips
                x_level =10
                y_level=10
                d_level=5
            elif point in range (17,23) and point %2 ==0: #righthand  finger tips
                x_level =10
                y_level=10
                d_level=5
            elif point in range (23,25):  # two hips do nothing
                x_level =1
                y_level=1
                d_level=1
            elif point ==25: #left kneeee 
                x_level =1
                y_level=1
                d_level=1

            elif point ==27: #left ankle
                x_level =1
                y_level=1
                d_level=1
            elif point ==29: #left heel
                x_level =1
                y_level=1
                d_level=1

            elif point ==31: #left index
                x_level =1
                y_level=1
                d_level=1


            elif point ==26: #rightt kneeee 
                x_level =1
                y_level=1
                d_level=1

            elif point ==28: #righhtt ankle
                x_level =1
                y_level=1
                d_level=1

            elif point ==30: #righhtt heel
                x_level =1
                y_level=1
                d_level=1

            elif point ==32: #righhtt index
                x_level =1
                y_level=1
                d_level=1
            
        x =int(x*x_level)
        y =int(y*y_level)
        d =int(d*d_level)
        
        npy_mat[point][0]=x
        npy_mat[point][1]=y
        npy_mat[point][2]=d
        
        point=point-1




    return npy1629

In [131]:
MAX_FRAME

156

In [132]:
def quantizeNpy (npy):
    # print('Q')
    myQLines=''
    npy_matrix=npy.reshape(-1,PER_FRAME_FEATURE)
    # print(npy_matrix.shape)
    for i in range(0,npy_matrix.shape[0]):
        # print(npy_matrix[i].shape)
        
        npy_matrix[i]=quantizeFrame(npy_matrix[i])
        
    return npy_matrix.flatten()

In [133]:
relativeQ_AUTSL226 ='../relativeQ/AUTSL226'
def getDirAndFN_AUTSL226(path):
    dir_path=relativeQ_AUTSL226
    label = path.split('/')
    i=0
    for i in range(2,len(label)-1):
        dir_path=dir_path+'/'+label[i]
    #print(dir_path)
    
    fileName =label[len(label)-1]
    return dir_path, fileName

In [135]:
def writeNpy(dir_path,file_name,npy):
    if os.path.exists(dir_path):
        pass
    else:
        os.makedirs(dir_path)

    np.save(dir_path+'/'+file_name, npy)

In [137]:
qpaths={'../NORMALIZED_RELATIVE_AUTSL226/train','../NORMALIZED_RELATIVE_AUTSL226/val','../NORMALIZED_RELATIVE_AUTSL226/test'}


for qpath in qpaths:
    trials=os.listdir(qpath)
    for trial in trials:
        trial_path=qpath+'/'+trial
        npy =np.load(trial_path)
        npy=quantizeNpy(npy)
        dest_dir,file_name=getDirAndFN_AUTSL226(trial_path)
        writeNpy(dest_dir,file_name,npy)

In [142]:
# npy=np.load('../relativeQ/AUTSL226/test/0_signer6_sample276.npy')
# npy=npy.reshape(-1,1629)
# print(npy.shape)
# print(npy[66])
# print(npy[67])
# print(npy[69])
# print(npy[155])

(156, 1629)
[2. 1. 1. ... 4. 6. 2.]
[2. 4. 2. ... 4. 4. 1.]
[2. 4. 2. ... 4. 4. 1.]
[2. 4. 2. ... 4. 4. 1.]
